SyntaxError: invalid syntax (3904524868.py, line 1)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
🎯 아이템 분석 대시보드 자동 생성기
새로운 CSV 데이터를 넣으면 자동으로 분석 대시보드가 생성됩니다.
"""

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, PieChart, Reference
import warnings
warnings.filterwarnings('ignore')

def load_data(file_path):
    """CSV 파일 로드"""
    # 다양한 인코딩 시도
    for encoding in ['utf-8', 'cp949', 'euc-kr', 'latin1']:
        try:
            df = pd.read_csv(file_path, encoding=encoding, skiprows=1)
            break
        except:
            continue
    
    # 컬럼명이 제대로 안 들어온 경우 수동 지정
    if len(df.columns) == 17:
        df.columns = [
            'Event_Category', 'Product_ID', 'Product_Price', 'Time', 'UserUUID',
            'a_attendance_count', 'a_user_name', 'a_user_stage_name', 'a_user_stage', 
            'b_diamond', 'b_ruby', 'boxAllCount', 'curserver', 'iap_platform',
            'install_day', 'total_payment_amount', 'user_rank'
        ]
    
    # 숫자형 변환
    numeric_cols = ['Product_Price', 'user_rank', 'total_payment_amount']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 결측치 및 이상값 제거
    df = df.dropna(subset=['Product_Price', 'UserUUID', 'user_rank'])
    df = df[df['Product_Price'] > 0]  # 음수 가격 제거
    df = df[df['user_rank'] > 0]  # 0 이하 랭크 제거
    
    return df

def apply_header_style(ws, row, start_col, end_col, color="4472C4"):
    """헤더 스타일"""
    for col in range(start_col, end_col + 1):
        cell = ws.cell(row=row, column=col)
        cell.font = Font(bold=True, color="FFFFFF", size=11)
        cell.fill = PatternFill(start_color=color, end_color=color, fill_type="solid")
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )

def apply_data_style(ws, start_row, end_row, start_col, end_col):
    """데이터 스타일"""
    for row in range(start_row, end_row + 1):
        for col in range(start_col, end_col + 1):
            cell = ws.cell(row=row, column=col)
            cell.border = Border(
                left=Side(style='thin'),
                right=Side(style='thin'),
                top=Side(style='thin'),
                bottom=Side(style='thin')
            )
            if col > start_col:
                cell.alignment = Alignment(horizontal='right')

def create_dashboard(input_file, output_file):
    """대시보드 생성"""
    
    print("=" * 70)
    print("📊 아이템 분석 대시보드 자동 생성기")
    print("=" * 70)
    
    # 1. 데이터 로드
    print("\n[1/6] 데이터 로딩 중...")
    df = load_data(input_file)
    total_sales = df['Product_Price'].sum()
    total_users = df['UserUUID'].nunique()
    total_orders = len(df)
    avg_price = df['Product_Price'].mean()
    
    print(f"   ✓ 데이터: {total_orders:,}건")
    print(f"   ✓ 유저: {total_users:,}명")
    print(f"   ✓ 총매출: {total_sales:,.0f}원")
    
    # 2. 분석
    print("\n[2/6] 데이터 분석 중...")
    
    # 랭크별
    df['rank_category'] = pd.cut(df['user_rank'], 
                                  bins=[0, 1000, 5000, 100000],
                                  labels=['고랭크(1-1000)', '중랭크(1001-5000)', '저랭크(5001+)'])
    
    rank_analysis = df.groupby('rank_category', observed=True).agg({
        'UserUUID': 'nunique',
        'Product_Price': ['sum', 'count', 'mean']
    }).round(0)
    rank_analysis.columns = ['유저수', '총매출', '구매수', '평균구매액']
    rank_analysis['매출비중(%)'] = (rank_analysis['총매출'] / rank_analysis['총매출'].sum() * 100).round(1)
    
    # 아이템별
    item_analysis = df.groupby('Product_ID').agg({
        'Product_Price': ['sum', 'count'],
        'UserUUID': 'nunique'
    }).round(0)
    item_analysis.columns = ['총매출', '판매수량', '구매유저수']
    item_analysis = item_analysis.sort_values('총매출', ascending=False).head(20)
    
    # 플랫폼별
    platform_analysis = df.groupby('iap_platform').agg({
        'UserUUID': 'nunique',
        'Product_Price': ['sum', 'count']
    }).round(0)
    platform_analysis.columns = ['유저수', '총매출', '구매수']
    
    # 가격대별
    df['price_range'] = pd.cut(df['Product_Price'], 
                                bins=[0, 500, 1000, 3000, 10000],
                                labels=['~500원', '501-1,000원', '1,001-3,000원', '3,001원~'])
    
    price_analysis = df.groupby('price_range', observed=True).agg({
        'Product_Price': ['sum', 'count'],
        'UserUUID': 'nunique'
    }).round(0)
    price_analysis.columns = ['총매출', '판매수량', '구매유저수']
    
    print("   ✓ 모든 분석 완료")
    
    # 3. 엑셀 생성
    print("\n[3/6] 엑셀 워크북 생성 중...")
    wb = Workbook()
    
    # === 시트 1: 대시보드 ===
    ws = wb.active
    ws.title = "📊 대시보드"
    
    ws['A1'] = "아이템 판매 분석 대시보드"
    ws['A1'].font = Font(size=18, bold=True, color="1F4E78")
    ws.merge_cells('A1:E1')
    ws['A1'].alignment = Alignment(horizontal='center')
    ws.row_dimensions[1].height = 30
    
    ws['A3'] = "📈 핵심 지표 (KPI)"
    ws['A3'].font = Font(size=13, bold=True)
    ws['A3'].fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
    ws.merge_cells('A3:E3')
    
    kpi_data = [
        ['지표', '값', '', '지표', '값'],
        ['총 매출액', f"{total_sales:,.0f} 원", '', '총 구매 유저', f"{total_users:,} 명"],
        ['총 구매건수', f"{total_orders:,} 건", '', '평균 구매금액', f"{avg_price:,.0f} 원"],
        ['최고 랭크', f"{int(df['user_rank'].min())}", '', '1인당 구매건수', f"{total_orders/total_users:.1f} 건"]
    ]
    
    for i, row_data in enumerate(kpi_data, start=4):
        for j, value in enumerate(row_data, start=1):
            ws.cell(row=i, column=j, value=value)
            if j in [1, 4]:
                ws.cell(row=i, column=j).font = Font(bold=True)
                ws.cell(row=i, column=j).fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid")
    
    ws.column_dimensions['A'].width = 18
    ws.column_dimensions['B'].width = 20
    ws.column_dimensions['C'].width = 3
    ws.column_dimensions['D'].width = 18
    ws.column_dimensions['E'].width = 20
    
    print("   ✓ 대시보드 시트")
    
    # === 시트 2: 랭크별 ===
    ws_rank = wb.create_sheet("🏆 랭크별")
    ws_rank['A1'] = "유저 랭크별 결제 분석"
    ws_rank['A1'].font = Font(size=14, bold=True)
    
    headers = ['랭크구분', '유저수', '총매출', '구매수', '평균구매액', '매출비중(%)']
    for j, h in enumerate(headers, 1):
        ws_rank.cell(row=3, column=j, value=h)
    apply_header_style(ws_rank, 3, 1, 6, "4472C4")
    
    row_idx = 4
    for idx, data in rank_analysis.iterrows():
        ws_rank.cell(row=row_idx, column=1, value=str(idx))
        ws_rank.cell(row=row_idx, column=2, value=int(data['유저수']))
        ws_rank.cell(row=row_idx, column=3, value=f"{int(data['총매출']):,}")
        ws_rank.cell(row=row_idx, column=4, value=int(data['구매수']))
        ws_rank.cell(row=row_idx, column=5, value=f"{int(data['평균구매액']):,}")
        ws_rank.cell(row=row_idx, column=6, value=f"{data['매출비중(%)']}%")
        row_idx += 1
    
    apply_data_style(ws_rank, 4, row_idx-1, 1, 6)
    ws_rank.column_dimensions['A'].width = 20
    
    print("   ✓ 랭크별 분석 시트")
    
    # === 시트 3: 아이템별 ===
    ws_item = wb.create_sheet("🎁 아이템별")
    ws_item['A1'] = "TOP 20 아이템 판매 분석"
    ws_item['A1'].font = Font(size=14, bold=True)
    
    headers = ['아이템명', '총매출', '판매수량', '구매유저수']
    for j, h in enumerate(headers, 1):
        ws_item.cell(row=3, column=j, value=h)
    apply_header_style(ws_item, 3, 1, 4, "70AD47")
    
    row_idx = 4
    for idx, data in item_analysis.iterrows():
        item_name = str(idx)[:50]
        ws_item.cell(row=row_idx, column=1, value=item_name)
        ws_item.cell(row=row_idx, column=2, value=f"{int(data['총매출']):,}")
        ws_item.cell(row=row_idx, column=3, value=int(data['판매수량']))
        ws_item.cell(row=row_idx, column=4, value=int(data['구매유저수']))
        row_idx += 1
    
    apply_data_style(ws_item, 4, row_idx-1, 1, 4)
    ws_item.column_dimensions['A'].width = 45
    
    print("   ✓ 아이템별 분석 시트")
    
    # === 시트 4: 플랫폼별 ===
    ws_platform = wb.create_sheet("📱 플랫폼별")
    ws_platform['A1'] = "플랫폼별 매출 분석"
    ws_platform['A1'].font = Font(size=14, bold=True)
    
    headers = ['플랫폼', '유저수', '총매출', '구매수']
    for j, h in enumerate(headers, 1):
        ws_platform.cell(row=3, column=j, value=h)
    apply_header_style(ws_platform, 3, 1, 4, "FFC000")
    
    row_idx = 4
    for idx, data in platform_analysis.iterrows():
        ws_platform.cell(row=row_idx, column=1, value=str(idx))
        ws_platform.cell(row=row_idx, column=2, value=int(data['유저수']))
        ws_platform.cell(row=row_idx, column=3, value=f"{int(data['총매출']):,}")
        ws_platform.cell(row=row_idx, column=4, value=int(data['구매수']))
        row_idx += 1
    
    apply_data_style(ws_platform, 4, row_idx-1, 1, 4)
    ws_platform.column_dimensions['A'].width = 20
    
    print("   ✓ 플랫폼별 분석 시트")
    
    # === 시트 5: 가격대별 ===
    ws_price = wb.create_sheet("💰 가격대별")
    ws_price['A1'] = "가격대별 구매 분석"
    ws_price['A1'].font = Font(size=14, bold=True)
    
    headers = ['가격대', '총매출', '판매수량', '구매유저수']
    for j, h in enumerate(headers, 1):
        ws_price.cell(row=3, column=j, value=h)
    apply_header_style(ws_price, 3, 1, 4, "C00000")
    
    row_idx = 4
    for idx, data in price_analysis.iterrows():
        ws_price.cell(row=row_idx, column=1, value=str(idx))
        ws_price.cell(row=row_idx, column=2, value=f"{int(data['총매출']):,}")
        ws_price.cell(row=row_idx, column=3, value=int(data['판매수량']))
        ws_price.cell(row=row_idx, column=4, value=int(data['구매유저수']))
        row_idx += 1
    
    apply_data_style(ws_price, 4, row_idx-1, 1, 4)
    ws_price.column_dimensions['A'].width = 20
    
    print("   ✓ 가격대별 분석 시트")
    
    # 4. 저장
    print("\n[4/6] 엑셀 파일 저장 중...")
    wb.save(output_file)
    print(f"   ✓ 저장 완료: {output_file}")
    
    # 5. 요약
    print("\n[5/6] 분석 결과 요약")
    print(f"   • 고랭크 유저: {int(rank_analysis.loc['고랭크(1-1000)', '유저수'])}명")
    print(f"   • 고랭크 매출비중: {rank_analysis.loc['고랭크(1-1000)', '매출비중(%)']}%")
    print(f"   • TOP 아이템: {item_analysis.index[0][:30]}")
    
    print("\n[6/6] 완료!")
    print("=" * 70)
    print(f"📄 생성된 파일: {output_file}")
    print(f"📊 시트 수: 5개 (대시보드, 랭크별, 아이템별, 플랫폼별, 가격대별)")
    print("\n💡 사용법:")
    print(f"   python {__file__} <새로운CSV> [출력파일명]")
    print("=" * 70)

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) < 2:
        print("\n사용법: python dashboard_final.py <CSV파일> [출력파일명]")
        print("예시: python dashboard_final.py 데이터.csv 결과.xlsx\n")
        sys.exit(1)
    
    input_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else "아이템분석_대시보드.xlsx"
    
    try:
        create_dashboard(input_file, output_file)
    except Exception as e:
        print(f"\n❌ 오류: {str(e)}")
        import traceback
        traceback.print_exc()
        sys.exit(1)


🔹 0행 3열 값 (출력용): 넨넨
이름넨넨
🔍 현재 작업 디렉토리: c:\JJG\파이썬
✅ 엑셀 데이터 로드 완료! 경로: C:\엑셀데이터\복사\TREASURE_DATA\넨넨.xlsx
✅ 로그인 완료!
✅ 페이지 로드 완료!
🔍 '다음' 버튼 탐색 중...
⚠️ '다음' 버튼을 찾지 못했습니다.
🔍 '확인' 버튼 탐색 중...
⚠️ 확인 버튼을 찾거나 클릭하지 못했습니다: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6af07a235
	0x7ff6aedd2630
	0x7ff6aeb616dd
	0x7ff6aebba27e
	0x7ff6aebba58c
	0x7ff6aec0ed77
	0x7ff6aec0baba
	0x7ff6aebab0ed
	0x7ff6aebabf63
	0x7ff6af0a5d60
	0x7ff6af09fe8a
	0x7ff6af0c1005
	0x7ff6aeded71e
	0x7ff6aedf4e1f
	0x7ff6aeddb7c4
	0x7ff6aeddb97f
	0x7ff6aedc18e8
	0x7ff95970257d
	0x7ff95a66af08

⏩ 그래도 계속 진행합니다.
🔹 0행 3열 값 (출력용): 넨넨
✅ 테이블 드롭다운 클릭 완료!
✅ BASE_DATA 선택 완료!
✅ 상세 검색 클릭 완료!
✅ 대상 클릭 완료!
✅ B2 값 입력 완료: j1
✅ 검색 버튼 클릭 완료!
✅ UUID 클릭 완료!
✅ 수정 버튼 클릭 완료!
✅ 웹 화면 25% 축소 완료!
🔍 기본 콘텐츠에서 요소 발견!
✅ 찾은 요소 개수: 3
🔹 매핑: hero_treasureChangeCount -> 요소 1
🔹 매핑: hero_treasureBaseSeed -> 요소 2
🔹 매핑: hero_treasureState -> 요소 3
🔍 처리 중: hero_treasureState
✅ Ace Editor 값 설정 완료: [
    [
        -1,
        -1,
        -